In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
SOURCE_TABLE = "fuel_project_dev.bronze.fuel_rates"
TARGET_TABLE = "fuel_project_dev.silver.fuel_rates"

In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
  fuel_station_id STRING,
  fuel_rate_id STRING,
  fuel_type STRING,
  fuel_cost DOUBLE,
  start_datetime TIMESTAMP,
  end_datetime TIMESTAMP,
  _silver_ingestion_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (fuel_station_id, fuel_type)   
    """
)

In [0]:
bronze_df = spark.table(SOURCE_TABLE)

In [0]:
from pyspark.sql.types import DoubleType
daily_df = (
    bronze_df
            .withColumn("_silver_ingestion_ts", current_timestamp())
            .withColumn("start_datetime", to_timestamp(col("start_datetime")))
            .withColumn("end_datetime", to_timestamp(col("end_datetime")))
            .withColumn("fuel_cost", col("fuel_cost").cast(DoubleType()))
            .filter(to_date(col("start_datetime")) >= date_sub(current_date(), 1))
            )

In [0]:
valid_fuel_types = ["Petrol", "Diesel", "CNG"]

valid_df = daily_df.filter(
    col("fuel_type").isin(valid_fuel_types) & col("fuel_cost").isNotNull()
)

invalid_df = daily_df.subtract(valid_df)

In [0]:
valid_df = valid_df.dropDuplicates([
    "fuel_station_id",
    "fuel_type",
    "start_datetime"
])

In [0]:
from delta.tables import DeltaTable

silver_table = DeltaTable.forName(
    spark, TARGET_TABLE
)

(
    silver_table.alias("t")
    .merge(
        valid_df.alias("s"),
        """
        t.fuel_station_id = s.fuel_station_id
        AND t.fuel_type = s.fuel_type
        AND t.start_datetime = s.start_datetime
        """
    )
    .whenMatchedUpdate(set={
        "fuel_cost": "s.fuel_cost",
        "end_datetime": "s.end_datetime",
        "_silver_ingestion_ts": "s._silver_ingestion_ts"
    })
    .whenNotMatchedInsert(values={
        "fuel_station_id": "s.fuel_station_id",
        "fuel_rate_id": "s.fuel_rate_id",
        "fuel_type": "s.fuel_type",
        "fuel_cost": "s.fuel_cost",
        "start_datetime": "s.start_datetime",
        "end_datetime": "s.end_datetime",
        "_silver_ingestion_ts": "s._silver_ingestion_ts"
    })
    .execute()
)
